# 행렬: 선형 변환 - 행렬 곱셈과 선형 변환

- Tutorial ID: `math-2-1`
- Tutorial: 행렬: 선형 변환
- Section ID: `math-2-1-1`
- Section: 행렬 곱셈과 선형 변환


In [ ]:
# ╔══════════════════════════════════════════════════════════════╗
# ║     행렬 곱셈과 선형 변환 — 개선판 v2                        ║
# ║     더 상세한 예제 + ★ 랭크(Rank) 개념 신규 추가             ║
# ╚══════════════════════════════════════════════════════════════╝
#
# 이 코드는 "정답을 한 번 실행"하는 용도가 아니라,
# 수학/아키텍처 개념이 실제 배열·텐서 연산으로 바뀌는 과정을
# 한 줄씩 추적하기 위한 실험 노트입니다.
#
# 📚 학습 목표:
#   1) 행렬 곱셈이 "함수 합성"이자 "공간 변환"임을 체감
#   2) shape 변화를 통해 데이터 흐름 추적
#   3) 랭크(rank)와 저랭크(low-rank)의 의미를 직접 실험으로 확인
#   4) 트랜스포머 어텐션이 저랭크 연산임을 수식으로 이해
#
# 📖 섹션 구성:
#   1.   행렬 곱셈 기초
#   2.   행렬-벡터 곱 = 선형 변환
#   3.   트랜스포머에서의 행렬 곱 (투영)
#   4.   중요한 행렬 성질
#   5.   트레이스와 프로베니우스 노름
#   5.5  ★[신규]★ 랭크(Rank)란 무엇인가? ← 6절 전에 꼭 읽으세요!
#   6.   저랭크 행렬 (Low-Rank)             ← 완전 개선
#
# ⚠️  주의:
#   - 숫자 하나하나보다 "shape 변화"와 "정보가 이동하는 방향"을 보세요.
#   - 이 파일은 NumPy만 사용합니다 (추가 설치 불필요).

In [ ]:
import numpy as np
print("NumPy 버전:", np.__version__)
print("모든 준비 완료! 아래 셀들을 순서대로 실행하세요.")

In [ ]:
# ─────────────────────────────────────────────────────────────────
# 1. 행렬 곱셈의 기초
# ─────────────────────────────────────────────────────────────────
#
# 행렬 곱셈 A @ B = C 에 대한 3가지 이해 방법:
#
#   [수식적] C[i, j] = A의 i번째 행과 B의 j번째 열의 내적(dot product)
#
#   [기하적] A와 B가 각각 "선형 변환"이라면,
#            A @ B 는 "B 먼저, 그다음 A를 적용"하는 합성 변환
#            즉: x → B(x) → A(B(x))
#
#   [흐름적] 데이터가 B 필터를 통과한 뒤, 다시 A 필터를 통과
#            x ──→ B ──→ A ──→ y

print("=" * 60)
print("1. 행렬 곱셈 (Matrix Multiplication)")
print("=" * 60)

# ─── 1-1. Shape 규칙 ───
# A: (m × k), B: (k × n)  →  C = A @ B: (m × n)
# 핵심: A의 '열 수' == B의 '행 수' (공통 차원 k)
# 이 k가 "내적을 계산할 때 항이 몇 개인지"를 결정합니다.

print("\\n■ 1-1. Shape 규칙: '가운데 차원'이 반드시 일치해야 합니다")
print("-" * 55)

m, k, n = 3, 4, 2
np.random.seed(0)
A_rect = np.random.randint(1, 5, (m, k))   # shape: (3, 4)
B_rect = np.random.randint(1, 5, (k, n))   # shape: (4, 2)
C_rect = A_rect @ B_rect                    # shape: (3, 2)

print(f"A: ({m}×{k}행렬)  @  B: ({k}×{n}행렬)  →  C: ({m}×{n}행렬)")
print(f"    └── 열 수={k} == 행 수={k} ✓ (가운데 차원이 일치)")
print(f"\\nA =\\n{A_rect}")
print(f"\\nB =\\n{B_rect}")
print(f"\\nC = A @ B =\\n{C_rect}")

# ─── 1-2. 원소별 직접 계산 ───
print("\\n■ 1-2. C[i,j]를 모두 직접 계산해 보기")
print("-" * 55)
print("  C[i,j] = A[i,:]의 각 원소 × B[:,j]의 각 원소를 곱하고 모두 더하기")
for i in range(m):
    for j in range(n):
        terms = " + ".join([f"{A_rect[i,t]}×{B_rect[t,j]}" for t in range(k)])
        result = C_rect[i, j]
        print(f"  C[{i},{j}] = {terms} = {result}")

# ─── 1-3. 정방 행렬 (기존 예제 유지 + 설명 보강) ───
print("\\n■ 1-3. 정방 행렬 (2×2) — 가장 기본적인 형태")
print("-" * 55)
A2 = np.array([[1, 2],
               [3, 4]])
B2 = np.array([[5, 6],
               [7, 8]])
C2 = A2 @ B2

print(f"A =\\n{A2}\\n")
print(f"B =\\n{B2}\\n")
print(f"A @ B =\\n{C2}")
print(f"\\n검증: C[1,1] = {A2[1,0]}×{B2[0,1]} + {A2[1,1]}×{B2[1,1]}")
print(f"              = {A2[1,0]*B2[0,1]} + {A2[1,1]*B2[1,1]}")
print(f"              = {A2[1,0]*B2[0,1]+A2[1,1]*B2[1,1]}  ← C[1,1] = {C2[1,1]} ✓")
print(f"\\n[실험 제안] A나 B의 값을 바꿔보세요. shape 규칙은 항상 성립합니다!")

In [ ]:
# ─────────────────────────────────────────────────────────────────
# 2. 행렬-벡터 곱 = 선형 변환
# ─────────────────────────────────────────────────────────────────
#
# "선형 변환(Linear Transformation)"이란?
# ──────────────────────────────────────
# 벡터 공간 V에서 다른 벡터 공간 W로 이동시키는 함수 f: V → W로,
# 두 성질을 만족합니다:
#   1) 덧셈 보존: f(u + v) = f(u) + f(v)  ("더하고 변환" = "변환하고 더함")
#   2) 스칼라 보존: f(c·v) = c·f(v)       ("늘리고 변환" = "변환하고 늘림")
#
# ★ 핵심: 모든 행렬 M은 하나의 선형 변환 f(v) = M @ v 를 정의합니다.
#          반대로, 모든 선형 변환은 행렬로 표현됩니다.
#
# 대표적인 선형 변환 종류: 회전 / 스케일 / 전단 / 반사 / 투영

print("=" * 60)
print("2. 행렬-벡터 곱 = 선형 변환")
print("=" * 60)

# 테스트에 사용할 벡터들
test_vectors = {
    "x축 단위벡터 (1,0)": np.array([1.0, 0.0]),
    "y축 단위벡터 (0,1)": np.array([0.0, 1.0]),
    "대각선 벡터  (1,1)": np.array([1.0, 1.0]),
}

# ─── 2-1. 회전 ───
print("\\n■ 2-1. 회전(Rotation) — 각도만큼 회전, 길이는 유지")
print("-" * 55)
#  cos(θ)  -sin(θ)
#  sin(θ)   cos(θ)
theta = np.pi / 4    # 45도 = π/4 라디안
R = np.array([[np.cos(theta), -np.sin(theta)],
              [np.sin(theta),  np.cos(theta)]])
print(f"45° 회전 행렬 R =\\n{np.round(R, 4)}")
print(f"\\n특징: 원점 중심 반시계 방향 45° 회전, 벡터 길이 불변")
for name, v in test_vectors.items():
    rv = R @ v
    print(f"  {name}: {v} → {np.round(rv, 4)},  길이: {np.linalg.norm(v):.3f} → {np.linalg.norm(rv):.3f}")

# 선형성 직접 검증
u, v = np.array([1.0, 0.0]), np.array([0.0, 1.0])
print(f"\\n[선형성 검증] R(u+v) == R(u) + R(v)?")
print(f"  R(u+v)    = {np.round(R @ (u+v), 4)}")
print(f"  R(u)+R(v) = {np.round(R@u + R@v, 4)}")
print(f"  일치?       {np.allclose(R @ (u+v), R@u + R@v)}")

# ─── 2-2. 스케일 ───
print("\\n■ 2-2. 스케일(Scale) — 각 축을 독립적으로 늘리거나 줄임")
print("-" * 55)
S = np.array([[3.0, 0.0],
              [0.0, 0.5]])
print(f"스케일 행렬 S =\\n{S}")
print(f"→ 대각 성분이 스케일 값: x 방향 3배, y 방향 0.5배")
for name, v in test_vectors.items():
    print(f"  S @ {name.split('(')[1].rstrip(')')} = {S @ np.array(eval('[' + name.split('(')[1].rstrip(')') + ']'))}")

# ─── 2-3. 전단 ───
print("\\n■ 2-3. 전단(Shear) — 공간을 기울이기 (직각이 변함)")
print("-" * 55)
# [1  k]   x → x + k*y
# [0  1]   y → y (고정)
k_shear = 1.5
H = np.array([[1.0, k_shear],
              [0.0, 1.0]])
print(f"수평 전단 행렬 H (k={k_shear}) =\\n{H}")
print(f"→ 변환 규칙: (x, y) → (x + {k_shear}·y,  y)")
for name, v in test_vectors.items():
    sv = H @ v
    print(f"  {name}: {v} → {sv}")
print("→ y가 클수록 x 방향으로 더 많이 밀립니다.")

# ─── 2-4. 반사 ───
print("\\n■ 2-4. 반사(Reflection) — 특정 축 기준 대칭")
print("-" * 55)
Fx = np.array([[1.0,  0.0],
               [0.0, -1.0]])   # x축 반사: y 부호 반전
Fy = np.array([[-1.0, 0.0],
               [ 0.0, 1.0]])   # y축 반사: x 부호 반전
print(f"x축 반사 행렬 Fx =\\n{Fx}")
for name, v in test_vectors.items():
    print(f"  Fx @ {name}: {v} → {Fx @ v}")
print(f"\\ny축 반사 행렬 Fy =\\n{Fy}")
for name, v in test_vectors.items():
    print(f"  Fy @ {name}: {v} → {Fy @ v}")

# ─── 2-5. 투영 ← 트랜스포머와 직결 ───
print("\\n■ 2-5. 투영(Projection) — 차원 축소  ★ 트랜스포머 핵심!")
print("-" * 55)
# 3차원 → 2차원 투영: z 성분을 버림
P = np.array([[1.0, 0.0, 0.0],
              [0.0, 1.0, 0.0]])   # shape: (2, 3)
v3d_list = [
    np.array([3.0, 4.0, 5.0]),
    np.array([1.0, 0.0, 9.0]),
    np.array([0.0, 2.0, 7.0]),
]
print(f"투영 행렬 P shape: (2, 3)  ← 3D를 2D로 줄이는 변환")
print(f"P =\\n{P}")
print()
for v3 in v3d_list:
    v2 = P @ v3
    print(f"  3D {v3} → 2D {v2}  (z={v3[2]:.0f} 사라짐)")

print(f"""
★ 이것이 트랜스포머 W_Q, W_K, W_V 연산의 본질입니다!

  X (각 토큰의 d_model 차원 벡터)
    @ W_Q (d_model × d_head 투영 행렬)
    = Q  (d_head 차원의 Query 벡터)

  d_model 차원 → d_head 차원으로 "투영"
  어떤 성분을 남길지는 W_Q, W_K, W_V 행렬이 학습으로 결정합니다.""")

In [ ]:
# ─────────────────────────────────────────────────────────────────
# 3. 트랜스포머에서의 행렬 곱 — 고차원 공간의 선형 투영
# ─────────────────────────────────────────────────────────────────
#
# 트랜스포머의 핵심 아이디어:
#   W_Q, W_K, W_V는 단순한 "숫자표"가 아니라
#   토큰 임베딩이 놓인 d_model 차원 공간을
#   어텐션 계산에 최적화된 d_head 차원 공간으로 보내는 선형 변환입니다.
#
#   X @ W_Q → 각 토큰을 "무엇을 묻는가(Query)" 공간으로 투영
#   X @ W_K → 각 토큰을 "어떤 정보를 가졌나(Key)" 공간으로 투영
#   X @ W_V → 각 토큰을 "가져올 내용(Value)" 공간으로 투영

print("=" * 60)
print("3. 트랜스포머 행렬 연산: 고차원 공간의 선형 투영")
print("=" * 60)

# ═══════════════════════════════════════════════
# [Step 1] 단일 문장의 투영 — 직관 형성
# ═══════════════════════════════════════════════
print("\\n▶ [Step 1] 단일 문장: 4차원 임베딩 → 2차원 Query 공간")
print("-" * 55)

seq_len = 3   # 토큰 개수: 예) ["I", "love", "AI"]
d_model = 4   # 각 토큰의 임베딩 차원 (실제 LLM: 768, 4096 등)
d_head  = 2   # 어텐션 헤드 차원 (눈으로 보기 위해 작게 설정)

print(f"하이퍼파라미터: seq_len={seq_len}, d_model={d_model}, d_head={d_head}")
print(f"(실제 모델: seq_len=2048~128k, d_model=768~8192, d_head=64~128)")

# X_single: shape (seq_len, d_model) = (3, 4)
# 각 행(row)이 토큰 하나를 나타냅니다.
# 예: "I love AI" 라는 문장의 3개 토큰
X_single = np.array([
    [ 1.0,  0.0,  0.5, -0.1],   # 토큰 0: "I"    의 4차원 임베딩
    [ 0.0,  1.0,  0.8,  0.2],   # 토큰 1: "love" 의 4차원 임베딩
    [ 0.0,  0.0,  0.2,  0.9],   # 토큰 2: "AI"   의 4차원 임베딩
])

# W_Q_single: shape (d_model, d_head) = (4, 2)
# 이 행렬이 "4차원 임베딩 공간 → 2차원 Query 공간"으로의 변환 규칙입니다.
# 각 열이 새로운 Query 축 하나를 정의합니다:
#   열 0: "Query의 첫 번째 차원을 어떻게 구성할까?"
#   열 1: "Query의 두 번째 차원을 어떻게 구성할까?"
W_Q_single = np.array([
    [ 0.5, -0.2],   # 임베딩의 1번째 성분이 각 Query 차원에 기여하는 방식
    [ 0.1,  0.8],   # 임베딩의 2번째 성분이 각 Query 차원에 기여하는 방식
    [-0.3,  0.4],   # 임베딩의 3번째 성분이 각 Query 차원에 기여하는 방식
    [ 0.0,  0.1],   # 임베딩의 4번째 성분이 각 Query 차원에 기여하는 방식
])

# 행렬 곱 shape 규칙:
#   X_single    @  W_Q_single
#   (3, 4)         (4, 2)
#         ↑─── 가운데 4가 일치! ───↑
#   결과: (3, 2)  ← 3개 토큰 × 2차원 Query
Q_single = X_single @ W_Q_single

print(f"\\nX_single.shape   = {X_single.shape}   ← (토큰 수, 임베딩 차원)")
print(f"W_Q_single.shape = {W_Q_single.shape}   ← (임베딩 차원, Query 차원)")
print(f"Q_single.shape   = {Q_single.shape}   ← (토큰 수, Query 차원)")
print(f"\\n입력 X (각 행 = 토큰 하나):\\n{X_single}")
print(f"\\nQuery 투영 행렬 W_Q:\\n{W_Q_single}")
print(f"\\nQuery 결과 Q = X @ W_Q:\\n{np.round(Q_single, 3)}")

# 한 토큰만 수동으로 계산해서 검증
print(f"\\n[검증] 토큰 0('I')의 Query 벡터를 수동 계산:")
for j in range(d_head):
    terms = " + ".join([
        f"({X_single[0,i]:.1f})×({W_Q_single[i,j]:.1f})"
        for i in range(d_model)
    ])
    val = sum(X_single[0,i] * W_Q_single[i,j] for i in range(d_model))
    print(f"  Q[0,{j}] = {terms} = {val:.3f}")
print(f"  → Q_single[0] = {np.round(Q_single[0], 3)}")
print(f"  → 3개 토큰 각각의 4차원 정보가 2차원 Query 벡터로 압축되었습니다!")

In [ ]:
# ═══════════════════════════════════════════════
# [Step 2] 배치(Batch) 처리 — 여러 문장 동시 변환
# ═══════════════════════════════════════════════
print("\\n▶ [Step 2] 배치(Batch) 처리: 여러 문장을 한 번에 변환")
print("-" * 55)
print("""
[왜 배치가 필요한가?]
  - GPU는 병렬 연산에 최적화되어 있습니다.
  - 문장 하나씩 처리하면 GPU의 병렬 능력이 낭비됩니다.
  - 여러 문장을 묶어서(배치) 한 번에 처리하면 훨씬 빠릅니다.
  - 이때 2D 행렬이 3D 텐서로 확장됩니다: (batch, seq, d_model)
""")

batch_size = 2   # 한 번에 처리할 문장 수

np.random.seed(42)   # 재현성: 이 seed로 실험 결과가 항상 동일

# X_batch: shape (batch_size, seq_len, d_model) = (2, 3, 4)
# 읽는 법:
#   X_batch[b]        → b번째 문장 전체         shape: (3, 4)
#   X_batch[b, t]     → b번째 문장의 t번째 토큰  shape: (4,)
#   X_batch[b, t, :]  → 그 토큰의 4차원 임베딩
X_batch = np.random.randn(batch_size, seq_len, d_model)

# W_Q: shape (d_model, d_head) = (4, 2)
# 실제 모델에서는 학습으로 최적화되는 파라미터
W_Q = np.random.randn(d_model, d_head) * 0.3

print(f"X_batch.shape = {X_batch.shape}   ← (문장 수, 토큰 수, 임베딩 차원)")
print(f"W_Q.shape     = {W_Q.shape}   ← (임베딩 차원, Query 차원)")

print(f"\\nX_batch[0] = 첫 번째 문장 (shape={X_batch[0].shape}):")
print(np.round(X_batch[0], 2))
print(f"\\nX_batch[1] = 두 번째 문장 (shape={X_batch[1].shape}):")
print(np.round(X_batch[1], 2))
print(f"\\nX_batch[0, 0] = 첫 번째 문장의 첫 번째 토큰 (shape={X_batch[0,0].shape}):")
print(np.round(X_batch[0, 0], 2))

# NumPy 고차원 행렬 곱 브로드캐스팅 규칙:
#   X_batch @ W_Q
#   (2, 3, 4)  @  (4, 2)
#
#   NumPy는 마지막 두 차원에서 행렬 곱을 수행합니다:
#     (3, 4) @ (4, 2) = (3, 2)
#   이것을 앞쪽 차원(2, ...)에 대해 반복합니다.
#   결과: (2, 3, 2)
#
#   즉, 아래 for 루프와 수학적으로 동일합니다:
#   Q_manual = np.zeros((2, 3, 2))
#   for b in range(2):
#       Q_manual[b] = X_batch[b] @ W_Q
Q_batch = X_batch @ W_Q

print(f"\\nQ_batch = X_batch @ W_Q")
print(f"Q_batch.shape = {Q_batch.shape}   ← (문장 수, 토큰 수, Query 차원)")
print(f"\\nQ_batch 전체:")
print(np.round(Q_batch, 3))

# 수동 계산으로 검증 (for 루프와 동일함을 확인)
Q_manual = np.zeros((batch_size, seq_len, d_head))
for b in range(batch_size):
    Q_manual[b] = X_batch[b] @ W_Q

print(f"\\n[검증] for 루프로 계산한 결과와 동일한가?")
print(f"  np.allclose(Q_batch, Q_manual) = {np.allclose(Q_batch, Q_manual)}")
print(f"\\n→ 반복문 없이, 행렬 곱셈 한 줄로")
print(f"   {batch_size}개 문장 × {seq_len}개 토큰 = {batch_size*seq_len}개 토큰이 동시에 변환됩니다!")
print(f"   배치 크기를 32, 64, 256으로 늘려도 코드 한 줄로 처리됩니다.")

In [ ]:
# ─────────────────────────────────────────────────────────────────
# 4. 중요한 행렬 성질
# ─────────────────────────────────────────────────────────────────

print("=" * 60)
print("4. 중요한 행렬 성질")
print("=" * 60)

A = np.array([[1, 2], [3, 4]])
B = np.array([[5, 6], [7, 8]])
C = np.array([[9, 0], [1, 2]])

# ─── 4-1. 단위 행렬 ───
print("\\n■ 4-1. 단위 행렬(Identity Matrix) — 곱해도 변하지 않음")
print("-" * 55)
I = np.eye(2)   # 2×2 단위 행렬: 대각선만 1, 나머지는 0
print(f"단위 행렬 I =\\n{I.astype(int)}")
print(f"\\nA @ I =\\n{A @ I}")
print(f"I @ A =\\n{I @ A}")
print(f"A @ I == A: {np.allclose(A @ I, A)},  I @ A == A: {np.allclose(I @ A, A)}")
print("\\n→ I는 숫자 1과 같은 역할: A × 1 = A 처럼 A @ I = A입니다.")
print("   트랜스포머에서 잔차 연결(Residual Connection)이 이와 관련됩니다.")

# ─── 4-2. 비가환성 ───
print("\\n■ 4-2. 비가환성(Non-Commutativity) — AB ≠ BA (일반적)")
print("-" * 55)
print(f"A =\\n{A}\\n")
print(f"B =\\n{B}")
print(f"\\nA @ B =\\n{A @ B}")
print(f"\\nB @ A =\\n{B @ A}")
print(f"\\nA @ B == B @ A? {np.allclose(A@B, B@A)}  ← 일반적으로 False!")
print("""
[직관] 왜 순서가 다르면 결과가 다를까요?
  A @ B: "B 변환 먼저 하고, 그 결과에 A 변환"
  B @ A: "A 변환 먼저 하고, 그 결과에 B 변환"
  → 변환 순서가 바뀌면 최종 결과가 달라집니다.

  실제 예:
  - "45° 회전한 다음 x축 스케일 2배" ≠ "x축 스케일 2배 한 다음 45° 회전"
""")

# 예외: 단위 행렬과는 교환 가능
print(f"[예외] 단위 행렬 I와는 교환 가능:")
print(f"  A @ I = {(A @ np.eye(2)).astype(int).tolist()}")
print(f"  I @ A = {(np.eye(2) @ A).astype(int).tolist()}")
print(f"  A @ I == I @ A: {np.allclose(A @ np.eye(2), np.eye(2) @ A)}")

# ─── 4-3. 결합법칙 ───
print("\\n■ 4-3. 결합법칙(Associativity) — (AB)C = A(BC)")
print("-" * 55)
left  = (A @ B) @ C
right = A @ (B @ C)
print(f"(A@B)@C =\\n{left}")
print(f"A@(B@C) =\\n{right}")
print(f"같은가? {np.allclose(left, right)}")
print("""
[의미] 세 개 이상의 행렬을 곱할 때 어떤 순서로 묶어 계산해도 결과가 같습니다.
  → 계산 효율을 위해 가장 빠른 묶음 순서를 선택할 수 있습니다!
  
  A: (m×k), B: (k×p), C: (p×n) 일 때:
  - (AB)C 방식: m·k·p + m·p·n 번의 곱셈
  - A(BC) 방식: k·p·n + m·k·n 번의 곱셈  ← 보통 이쪽이 더 효율적
""")

# ─── 4-4. 전치의 성질 ───
print("■ 4-4. 전치(Transpose) — (AB)^T = B^T A^T (순서 뒤집힘!)")
print("-" * 55)
lhs = (A @ B).T
rhs = B.T @ A.T
print(f"(A@B)^T =\\n{lhs}")
print(f"B^T @ A^T =\\n{rhs}")
print(f"같은가? {np.allclose(lhs, rhs)}")
print(f"\\n[주의] (AB)^T ≠ A^T @ B^T  ← 순서가 뒤집힘!")
print(f"  잘못된 식: {np.round(A.T @ B.T, 0).astype(int).tolist()}")
print(f"  올바른 식: {rhs.astype(int).tolist()}")

# ─── 4-5. 역행렬 ───
print("\\n■ 4-5. 역행렬(Inverse) — A @ A⁻¹ = I  (변환을 되돌림)")
print("-" * 55)
A_inv = np.linalg.inv(A)
print(f"A =\\n{A}")
print(f"\\nA⁻¹ =\\n{np.round(A_inv, 4)}")
print(f"\\nA @ A⁻¹ =\\n{np.round(A @ A_inv, 4)}  ← 단위 행렬에 매우 가까움")
print(f"\\n[직관] 45° 회전 행렬의 역행렬 = -45° 회전 행렬:")
theta = np.pi/4
R_pos = np.array([[np.cos(theta), -np.sin(theta)],
                  [np.sin(theta),  np.cos(theta)]])
R_neg = np.array([[np.cos(-theta), -np.sin(-theta)],
                  [np.sin(-theta),  np.cos(-theta)]])
print(f"  R(45°)⁻¹ == R(-45°)? {np.allclose(np.linalg.inv(R_pos), R_neg)}")
print(f"  R(45°) @ R(-45°) == I? {np.allclose(R_pos @ R_neg, np.eye(2))}")
print(f"\\n[주의] 모든 행렬에 역행렬이 있지는 않습니다!")
print(f"  rank < n 인 행렬 (저랭크 행렬)은 역행렬이 존재하지 않습니다.")
print(f"  (이것이 6절의 저랭크 행렬과 연결됩니다!)")

In [ ]:
# ─────────────────────────────────────────────────────────────────
# 5. 트레이스(Trace)와 프로베니우스 노름(Frobenius Norm)
# ─────────────────────────────────────────────────────────────────

print("=" * 60)
print("5. 트레이스(Trace)와 프로베니우스 노름(Frobenius Norm)")
print("=" * 60)

M = np.array([[1, 2, 3],
              [4, 5, 6],
              [7, 8, 9]])
A = np.array([[1, 2], [3, 4]])
B = np.array([[5, 6], [7, 8]])

# ─── 5-1. 트레이스 ───
print("\\n■ 5-1. 트레이스(Trace) — 정방 행렬의 대각 원소 합")
print("-" * 55)
print(f"M =\\n{M}")
trace_M = np.trace(M)
print(f"\\ntrace(M) = M[0,0] + M[1,1] + M[2,2]")
print(f"         = {M[0,0]} + {M[1,1]} + {M[2,2]} = {trace_M}")

print(f"\\n[응용] 트레이스는 고유값(eigenvalue)의 합과 같습니다:")
eigenvalues = np.linalg.eigvals(M)
print(f"  M의 고유값들: {np.round(eigenvalues, 4)}")
print(f"  고유값 합:    {np.round(np.sum(eigenvalues).real, 4)}")
print(f"  trace(M):     {trace_M}")
print(f"  같은가?       {np.allclose(trace_M, np.sum(eigenvalues).real)}")

print(f"\\n■ 트레이스의 순환 불변성: tr(AB) = tr(BA)")
print(f"  tr(A@B) = {np.trace(A@B)}")
print(f"  tr(B@A) = {np.trace(B@A)}")
print(f"  같은가? {np.allclose(np.trace(A@B), np.trace(B@A))}")
print(f"  → AB ≠ BA 이지만, 트레이스값은 순서에 무관!")
print(f"  → 이 성질은 손실 함수 이론 분석에서 자주 활용됩니다.")

# ─── 5-2. 프로베니우스 노름 ───
print("\\n■ 5-2. 프로베니우스 노름 — 행렬의 '전체 크기'를 측정")
print("-" * 55)
print("""
[개념] 프로베니우스 노름은 벡터 노름의 행렬 버전입니다.

  벡터 노름:   ||v||₂ = √(v₁² + v₂² + v₃² + ...)
  행렬 노름:   ||M||_F = √(모든 원소를 제곱하고 더한 후 제곱근)
                        = √(M₁₁² + M₁₂² + ... + Mₙₘ²)

[직관]
  - 행렬의 "전체 에너지" 또는 "신호 강도"를 하나의 수로 표현
  - ||M||_F = 0 이면 M은 영행렬

[응용]
  1. 두 행렬이 얼마나 다른지 측정: ||A - B||_F
     예) 원본 모델과 압축 모델의 차이
  2. 가중치 정규화: L2 정규화에서 ||W||_F 를 패널티로 사용
  3. 저랭크 근사 품질 평가 (SVD 활용)
""")

print(f"M =\\n{M}")

# 3단계 계산 과정 시각화
M_sq = M ** 2
sum_sq = np.sum(M_sq)
frob_manual = np.sqrt(sum_sq)
frob_numpy  = np.linalg.norm(M, 'fro')

print(f"\\n[1단계] 각 원소를 제곱합니다: M² =\\n{M_sq}")
elements = M_sq.flatten()
print(f"\\n[2단계] 제곱된 원소들을 모두 더합니다:")
print(f"  {'  +  '.join(str(x) for x in elements)}")
print(f"  = {sum_sq}")
print(f"\\n[3단계] 합계의 제곱근을 구합니다:")
print(f"  √{sum_sq} ≈ {frob_manual:.6f}")
print(f"\\nnp.linalg.norm(M, 'fro') = {frob_numpy:.6f}")
print(f"수동 계산과 일치? {np.allclose(frob_manual, frob_numpy)}")

# 두 행렬 차이 측정 예시
A_orig   = np.random.randn(4, 4)
noise    = np.random.randn(4, 4) * 0.1
A_noisy  = A_orig + noise
diff_frob = np.linalg.norm(A_orig - A_noisy, 'fro')
print(f"\\n[응용 예시] 원본 행렬과 노이즈 추가 행렬의 차이:")
print(f"  ||A_orig - A_noisy||_F = {diff_frob:.4f}")
print(f"  → 이 값이 작을수록 두 행렬이 비슷합니다.")

In [ ]:
# ═══════════════════════════════════════════════════════════════
# 5.5  ★ 랭크(Rank)란 무엇인가? — 6절을 보기 전에 꼭 읽으세요!
# ═══════════════════════════════════════════════════════════════
#
# 기존 노트북은 6절에서 설명 없이 갑자기 "rank"를 사용했습니다.
# 이 섹션이 그 부분을 완전히 보완합니다.
#
# 랭크는 선형대수에서 가장 핵심적인 개념 중 하나입니다.
# 6절의 "저랭크 행렬"을 이해하려면 이 섹션이 반드시 필요합니다.

print("=" * 60)
print("5.5 ★ 랭크(Rank)란 무엇인가?")
print("    6절(저랭크 행렬)을 위한 필수 선행 개념")
print("=" * 60)

print("""
─────────────────────────────────────────────────────────────
[핵심 직관] 랭크 = "행렬이 실제로 표현하는 독립적인 방향의 수"
─────────────────────────────────────────────────────────────

행렬은 선형 변환(공간 변환)입니다.
어떤 변환은 입력 공간을 "온전히" 보존하고,
어떤 변환은 공간을 "납작하게 찌그러뜨립니다."

  rank = n (full rank):  n차원 공간 → n차원 공간 (정보 보존)
  rank = 2:              아무리 고차원 입력도 → 2차원 평면으로 납작해짐
  rank = 1:              아무리 고차원 입력도 → 1차원 직선 위로 찌그러짐

[다른 정의]
  rank = 행렬의 열(column)들 중 "선형 독립"인 열의 최대 개수
  rank = 행렬의 행(row)들 중  "선형 독립"인 행의 최대 개수
  (신기하게도 행 기준과 열 기준 모두 같은 값!)

[선형 독립이란?]
  어떤 벡터가 다른 벡터들의 선형 결합(스칼라 배수의 합)으로
  만들어지지 않을 때 "독립적"이라고 합니다.
  쉽게 말해: "새로운 방향 정보"를 가진 벡터.
  예) (1,0)과 (2,0)은 하나가 다른 하나의 2배 → 독립적이지 않음
      (1,0)과 (0,1)은 어느 쪽도 다른 쪽의 배수가 아님 → 독립적
""")

# ─── 5.5.1 Rank-1 행렬 ───
print("■ 5.5.1 Rank-1 행렬 — 외적(Outer Product)으로 만들기")
print("-" * 55)
print("""
[외적 (Outer Product)]
  두 열벡터 u (n×1), v (m×1)의 외적:
    u @ v.T  →  shape (n, m)
  결과의 특징: 모든 행이 v.T의 스칼라 배수입니다.
  → 오직 1가지 방향만 존재 → rank = 1
""")

u = np.array([[2], [3], [4]])   # 3×1 열벡터
v = np.array([[1], [5]])         # 2×1 열벡터
R1 = u @ v.T                     # shape: (3, 2)

print(f"u = {u.flatten()} (shape {u.shape}의 열벡터)")
print(f"v = {v.flatten()} (shape {v.shape}의 열벡터)")
print(f"\\nR1 = u @ v.T =")
print(R1)
print(f"shape: {R1.shape},  rank = {np.linalg.matrix_rank(R1)}")
print()
print("[관찰] R1의 각 행이 v.T의 스칼라 배수인지 확인:")
for i in range(R1.shape[0]):
    coeff = u[i, 0]
    expected = coeff * v.flatten()
    match = np.allclose(R1[i], expected)
    print(f"  R1[{i}] = {R1[i]}  =  {coeff} × {v.flatten()}  ✓ {match}")
print()
print("→ 모든 행이 v 방향만 가리킵니다. 독립적인 방향이 1개뿐 → rank = 1!")

# 더 명확한 rank-1 예시
print("\\n[예시] 아래 행렬의 rank를 예측해보세요:")
M_ex = np.array([[1, 2, 3],
                 [2, 4, 6],    # 행 0의 2배
                 [3, 6, 9]])   # 행 0의 3배
print(M_ex)
print(f"  → 행 1 = 2 × 행 0,  행 2 = 3 × 행 0")
print(f"  → 독립적인 행은 1개뿐 → rank = {np.linalg.matrix_rank(M_ex)}")

In [ ]:
# ─── 5.5.2 랭크를 1씩 쌓아 올리기 ───
print("\\n■ 5.5.2 랭크를 1씩 쌓아 올리기 — rank-k 행렬의 구조")
print("-" * 55)
print("""
[핵심] rank-k 행렬 = rank-1 행렬 k개의 합!

  M = σ₁ u₁v₁ᵀ  +  σ₂ u₂v₂ᵀ  +  ...  +  σₖ uₖvₖᵀ

  각 항 σᵢ uᵢvᵢᵀ 가 하나의 "독립적 방향"을 추가합니다.
  이것이 SVD(특이값 분해)의 핵심 구조입니다.
""")

n = 5   # 5×5 행렬 실험
e = np.eye(n)   # 표준 기저 벡터들 (직교하므로 모두 독립)

# 단계별로 rank-1 행렬을 더해 rank를 높임
M_r1 = np.outer(e[0], e[0])                           # rank 1
M_r2 = np.outer(e[0], e[0]) + np.outer(e[1], e[1])   # rank 2
M_r3 = M_r2 + np.outer(e[2], e[2])                   # rank 3
M_r5 = np.eye(n)                                       # rank 5 (full rank)

for label, M_ex in [("Rank-1", M_r1), ("Rank-2", M_r2),
                     ("Rank-3", M_r3), ("Rank-5 (full rank)", M_r5)]:
    r = np.linalg.matrix_rank(M_ex)
    filled = "█" * r + "░" * (n - r)
    print(f"  {label}: rank={r}  [{filled}]  ({n}×{n} 행렬)")
    print(f"  \\n{M_ex.astype(int)}\\n")

print("""
[해석]
  rank = 1: 5D 공간의 모든 입력을 1D 직선으로 찌그러뜨림
  rank = 2: 5D 공간의 모든 입력을 2D 평면으로 찌그러뜨림
  rank = 5: 5D → 5D, 정보 손실 없음 (full rank)

  rank가 낮을수록 → 더 많은 정보가 압축/손실됩니다.
""")

# ─── 5.5.3 ★★ 핵심: 행렬 곱의 병목이 랭크를 제한합니다 ───
print("■ 5.5.3 ★★ 핵심: 행렬 곱의 '병목'이 랭크를 제한합니다 ★★")
print("   이 개념이 6절 저랭크 행렬의 핵심입니다!")
print("-" * 55)
print("""
[정리] rank(A @ B) ≤ min(rank(A), rank(B))

더 중요한 사실:
  A: (n × r), B: (r × n) 일 때
  A @ B:  (n × n) 크기지만,  rank ≤ r

  이유:
    A: n차원 → r차원 (압축, 정보 일부 손실)
    B: r차원 → n차원 (복원 시도)
    
    하지만 r차원으로 압축한 순간 이미 정보가 손실되었습니다!
    아무리 크게 펼쳐도 잃어버린 정보는 돌아오지 않습니다.
""")

np.random.seed(42)
n_dim, r_bottleneck = 6, 2

A_bn = np.random.randn(n_dim, r_bottleneck)   # (6, 2): 6차원 → 2차원 압축
B_bn = np.random.randn(r_bottleneck, n_dim)   # (2, 6): 2차원 → 6차원 복원 시도
AB_bn = A_bn @ B_bn                            # (6, 6): 크기는 6×6

print(f"A.shape = {A_bn.shape}   rank(A) = {np.linalg.matrix_rank(A_bn)}")
print(f"B.shape = {B_bn.shape}   rank(B) = {np.linalg.matrix_rank(B_bn)}")
print(f"\\nA @ B:  shape = {AB_bn.shape}   (크기는 {n_dim}×{n_dim}!)")
print(f"        rank  = {np.linalg.matrix_rank(AB_bn)}        (하지만 rank는 {r_bottleneck}뿐)")
print()
print("┌─────────────────────────────────────────────────────┐")
print("│  비유: 물이 좁은 파이프를 통과하는 것과 같습니다   │")
print("│                                                     │")
print(f"│   6D ─────→ 2D ─────→ 6D                          │")
print(f"│   [A압축]  ↑ 병목  [B복원]                         │")
print(f"│           여기서 4차원 분량의 정보 손실!           │")
print("│                                                     │")
print("│   아무리 뒤에서 6차원으로 펼쳐도                   │")
print("│   잃어버린 정보는 돌아오지 않습니다.               │")
print("└─────────────────────────────────────────────────────┘")

# SVD로 랭크 구조를 눈으로 확인
_, singular_vals, _ = np.linalg.svd(AB_bn)
print(f"\\n[SVD로 확인] AB의 특이값(singular values):")
print(f"  {np.round(singular_vals, 6)}")
significant = np.sum(singular_vals > 1e-10)
print(f"  → 0이 아닌 특이값 개수 = {significant}  ← 이것이 rank!")
print(f"  → 이론적 상한 rank({r_bottleneck}) = {r_bottleneck}과 일치합니다.")

In [ ]:
# ═══════════════════════════════════════════════════════════════
# 6. 저랭크 행렬 (Low-Rank Matrices) — 트랜스포머 어텐션의 핵심
# ═══════════════════════════════════════════════════════════════
#
# [사전 지식 확인]
# 이 섹션은 5.5절의 "병목이 랭크를 제한한다"는 개념을 직접 활용합니다.
# 5.5절을 먼저 이해하고 오셨다면 이 섹션이 훨씬 명확하게 보일 것입니다.
#
# [이 섹션에서 배울 것]
# 1. W_QK = W_Q @ W_K^T 가 왜 저랭크 행렬인가
# 2. 이것이 어텐션 헤드의 '용량'을 어떻게 결정하는가
# 3. d_head를 바꾸면 어떤 변화가 생기는가
# 4. [보너스] LoRA가 이 원리를 어떻게 활용하는가

print("=" * 60)
print("6. 저랭크 행렬 (Low-Rank) — 어텐션 헤드의 '용량(Capacity)'")
print("=" * 60)

print("""
[어텐션 연산에서 W_QK가 등장하는 맥락]
─────────────────────────────────────────────────────────────
어텐션에서 다음과 같은 연산이 일어납니다:

  Q = X @ W_Q     (X: 입력 토큰 행렬, W_Q: Query 투영)
  K = X @ W_K     (W_K: Key 투영)

  어텐션 점수 = (Q @ K^T) / √d_head
             = (X @ W_Q) @ (X @ W_K)^T / √d_head
             = X @ W_Q @ W_K^T @ X^T / √d_head
                      ────────────
                 이 부분 = W_QK = W_Q @ W_K^T
                 shape: (d_model × d_model)

5.5.3에서 배운 '병목' 개념으로 W_QK를 분석하면:

  W_Q:    (d_model × d_head)   ← d_head 차원 병목
  W_K^T:  (d_head  × d_model)  ← 다시 d_model 차원으로

  → W_QK의 rank ≤ d_head  (병목 크기가 rank를 제한!)
""")

# ─── 6-1. 수치 실험으로 확인 ───
print("\\n■ 6-1. 수치 실험: W_QK의 rank 직접 확인")
print("-" * 55)

d_model = 8   # 임베딩 차원 (실제: 768, 4096, 8192 등)
d_head  = 2   # 어텐션 헤드 차원 (병목 크기)

np.random.seed(42)
W_Q = np.random.randn(d_model, d_head)   # shape: (8, 2)
W_K = np.random.randn(d_model, d_head)   # shape: (8, 2)
W_QK = W_Q @ W_K.T                       # shape: (8, 8)

print(f"d_model = {d_model}   (임베딩 차원)")
print(f"d_head  = {d_head}    (어텐션 헤드 차원 = 병목 크기)")
print()
print(f"W_Q.shape  = {W_Q.shape}   ← {d_model}차원 → {d_head}차원 압축 (Query 투영)")
print(f"W_K.shape  = {W_K.shape}   ← {d_model}차원 → {d_head}차원 압축 (Key 투영)")
print()
print(f"W_QK = W_Q @ W_K.T")
print(f"W_QK.shape = {W_QK.shape}  ← {d_model}×{d_model}={d_model*d_model}개 원소를 가진 큰 행렬")

rank_WQK = np.linalg.matrix_rank(W_QK)
print(f"\\nrank(W_QK) = {rank_WQK}")
print(f"  ↑ {d_model}×{d_model}=64개 숫자가 있지만, 독립적인 방향은 {rank_WQK}개뿐!")
print(f"  이론적 상한: min(d_head, d_head) = {d_head}  ← 병목이 결정!")

# SVD로 특이값 시각화
_, sv, _ = np.linalg.svd(W_QK)
print(f"\\n[SVD 확인] W_QK의 특이값들:")
for i, s in enumerate(sv):
    bar = "█" * int(s * 2) if s > 1e-10 else ""
    flag = " ← 유의미" if s > 1e-10 else " ← 거의 0"
    print(f"  σ[{i}] = {s:.4f}  {bar}{flag}")
print(f"\\n0이 아닌 특이값 = {np.sum(sv > 1e-10)}개 = rank({rank_WQK})와 일치!")

# ─── 6-2. 어텐션 헤드의 '용량' ───
print("""
\\n■ 6-2. 어텐션 헤드의 '용량(Capacity)'
─────────────────────────────────────────────────────────────
W_QK의 rank = d_head 가 의미하는 것:

  어텐션 헤드 하나는 토큰 간 관계를 찾을 때
  전체 d_model 차원 공간이 아니라
  오직 d_head 차원의 "부분공간"만 탐색합니다.

  d_head = 2  → 2가지 독립적 패턴만 포착 가능 (용량 작음)
  d_head = 64 → 64가지 독립적 패턴 포착 가능 (용량 큼)

  d_head 크면: 더 복잡한 관계 포착, 계산 비용 ↑
  d_head 작으면: 단순한 관계만 포착, 계산 비용 ↓
""")

# ─── 6-3. d_head를 바꿔보는 실험 ───
print("■ 6-3. d_head를 변경하며 rank 변화 관찰")
print("-" * 55)
for dh in [1, 2, 4, 8]:
    np.random.seed(42)
    WQ = np.random.randn(d_model, dh)
    WK = np.random.randn(d_model, dh)
    WQK = WQ @ WK.T
    r = np.linalg.matrix_rank(WQK)
    pct = r / d_model * 100
    bar_filled = "█" * r
    bar_empty  = "░" * (d_model - r)
    print(f"  d_head={dh:2d}  →  rank(W_QK)={r}  "
          f"[{bar_filled}{bar_empty}]  {pct:.0f}% 용량 활용")

print(f"\\n  d_head={d_model}이 되어야 full rank(={d_model})가 됩니다.")
print(f"  실제 모델: d_head << d_model (저랭크 구조가 의도적으로 설계됨)")

# ─── 6-4. 왜 저랭크 구조가 유용한가? ───
print("""
\\n■ 6-4. 왜 저랭크 구조가 유용한가?
─────────────────────────────────────────────────────────────
[1] 계산 효율
  - Q, K, V를 직접 d_model×d_model로 계산하면 비용이 큼
  - d_head 차원을 거치면 훨씬 적은 파라미터로 동일한 효과

[2] 귀납적 편향(Inductive Bias)
  - 저랭크 구조는 "관련된 몇 가지 패턴에만 집중"하도록 강제
  - 이것이 오히려 일반화(generalization)에 도움됨 (오버피팅 방지)

[3] 멀티 헤드(Multi-Head Attention)와의 시너지
  - 헤드 하나: rank = d_head 수준의 정보
  - 헤드 h개:  h × d_head 수준의 정보 (각 헤드가 다른 패턴 담당)
  - 다양한 관점에서 토큰 관계를 동시에 포착!
""")

# ─── 6-5. [보너스] LoRA: 저랭크 분해의 응용 ───
print("■ 6-5. [보너스] LoRA — 저랭크 분해를 이용한 효율적 파인튜닝")
print("-" * 55)
print("""
[LoRA: Low-Rank Adaptation of Large Language Models]

대규모 LLM을 특정 과업(task)에 맞게 파인튜닝할 때:

  기존 방법: 모든 W (d×d) 파라미터를 업데이트 → d² 파라미터 필요
  LoRA 방법: ΔW = A @ B  (A: d×r, B: r×d,  r << d)만 학습
              → 2·d·r 개의 파라미터만 필요

핵심 가정: "파인튜닝에 필요한 변화 ΔW는 저랭크 구조를 가진다"
  이 가정이 유효한 이유? 원래 W_Q, W_K 자체가 저랭크이기 때문!
""")

d = 8   # d_model (예시)
r = 2   # LoRA rank

# 파라미터 수 비교
params_full = d * d
params_lora = d * r + r * d
reduction   = 1 - params_lora / params_full

print(f"d_model = {d},  r (LoRA rank) = {r}")
print()
print(f"기존 방법: W의 파라미터 수 = {d}×{d} = {params_full}")
print(f"LoRA 방법: A({d}×{r}) + B({r}×{d}) = {d*r} + {r*d} = {params_lora} 파라미터")
print(f"절감율: {reduction:.0%} ({params_full - params_lora}개 파라미터 절약)")

np.random.seed(1)
A_lora = np.random.randn(d, r) * 0.01   # 작은 값으로 초기화
B_lora = np.zeros((r, d))                # 0으로 초기화 (처음엔 ΔW = 0)
delta_W = A_lora @ B_lora

print(f"\\nA_lora.shape = {A_lora.shape}  (학습 파라미터)")
print(f"B_lora.shape = {B_lora.shape}  (학습 파라미터, 0으로 초기화)")
print(f"ΔW = A @ B: shape = {delta_W.shape}, rank = {np.linalg.matrix_rank(delta_W + np.eye(d)*1e-10)}")
print()
print("[요약]")
print(f"  LoRA는 '저랭크 행렬로도 충분히 좋은 파인튜닝이 된다'는")
print(f"  가설을 활용합니다.")
print(f"  이 가설이 작동하는 이유 중 하나는")
print(f"  W_Q, W_K 자체가 이미 저랭크 구조를 가지기 때문입니다!")
print()
print("=" * 60)
print("실습 완료! 다음 단계: d_head, d_model, r 값을 바꾸며 실험해보세요.")
print("=" * 60)